# CUESTIONARIO

## 1. ¿Qué es FAERS y cuál es su utilidad en farmacovigilancia?

FAERS es una base de datos administrada por la FDA, contiene reportes espontaneos de eventos adversos.  
Contiene informacion sobre eventos adversos, medicamentos, productos biologicos y errores de medicacion.  
Un evento adverso es un  acontecimiento no deseado ocurrido durante el uso de un medicamento.  
Una asociacion encontrada en FAERS no implica causalidad porque los reportes son espontaneos, es decir, que no son controlados, cualquiera puede reportar una sospecha sin que exista un diseño experimental que aisle causas posibles

## 2. ¿Cómo está organizado un reporte dentro de los archivos XML de FAERS?

Usamos la extension XML para almacenar información mediante anidados, es decir, primero está contenido en un elemento \<safetyreport>, dentro del él existe \<patient> y dentro de este puede haber multiples elementos de \<drug> y \<reaction>.  
Por eso mismo safetyreportid puede conteneer varios medicamentos y varias reacciones, es una jerarquia donde un paciente puede haber tomado varios farmacos y presentado varias reacciones simultaneamente

## 3. Explica el significado de las siguientes variables

* safetyreportid; identificador unico del reporte de seguridad, esta relacionada a medicamentos y reacciones
* safetyreportversion; numero de version del reporte
* occurcountry; pais donde ocurrio el evento reportado
* medicinalproduct; nombre comercial del medicamento
* activesubstancename; principio activo
* drugcharacterization; caracterizacion del farmaco(sospechoso, etc)
* reactionmeddrapt; identidica la reacion adversa 
* receivedate;fecha histórica inicial del caso
* receiptdate. fecha de recepción de la versión del reporte  
Para el análisis geográfico la mas importante es occurcountry, para el análisis temporal es receiptdate, para el análisis medicamento-evento las claves serían:activesubstancename/medicinalproduct que forman drug key y reactionmeddrapt

## 4. ¿Cuál es la diferencia entre medicinalproduct y activesubstancename?

medicinalproduct es el nombre del producto tal y como fue reportado que suele seer el nombre ccomercial  y activesubstancename es el principio activo que contiene el producto. Un producto activo puede venderse bajo distintas marcas como por ejemplo:  
DUPIXENT y DUPILUMAB o REVLIMID y LENALIDOMIDE.  
Entonces si el análisis se hiciera con medicinalproduct notamos que el mismo farmaco quedaria fragmentado en varias categorias (por cada nombre comercial), por lo tanto disminuiria la señal estadística. Por eso se usa drug key priorizando activesubstancename, ya que esto agrupa  correctamente todos los reportes del mismo farmaco independientemente de la marca usada. 

## 5. ¿Qué papel tiene drugcharacterization en el proyecto?

Indica el rol que juega cada medicamento respecto al evento adverso reportado  

| Codigo | Papel |
| :---: | :--- |
| **1** | **Suspect** |
| **2** | **Concomitant** |
| **3** | **Interacting** |
| **4** | **Drug not adminitered** |  

Que un medicamento diga Suspect significa que fue señalado como reaccion.  
Concomitant significa que el paciente lo tomaba tambien pero no fue señalado de esa manera. Por lo que no sería correcto tratar a todos los medicamentos de un reporte por igual, por eso se habla del evento adverso para evitar señales o ruido.  
El analisis principal se usa **drugcharacterization = 1**

## 6. ¿Por qué es necesario deduplicar los medicamentos dentro de un reporte?

Las 2938 filas \<drug> con **drugcharacterization = Suspect** no equivalen a 2938 exposiciones independientes porque un mismo medicamento puede aparecer varias veces dentro del mismo reporte, ya sea por diferencias de dosis, duracion e indicacion. Al agrupar por \(safetyreportid, drug_key) se obtiene 1589 pares unicos.  
Si se usara 2938 filas de forma independiente los datos serian redundantes.  
En el caso de TOCILIZUMAB que aparece 85 veces es un mismo reporte, nos dice que probablemente corresponde a multiples registros de administracion del mismo farmaco en el mismo paciente, no a 85 pacientes ni 85 exposiciones distintas.

## 7. ¿Cuál es la unidad analítica definida para los medicamentos?

La unidad analitica es la combinacion de \(safereportid, drug_key). Garantiza que cada medicamento contribuya como maximo una vez por reporte, evitando el sobreconteo.   
Al incroporar las reacciones, entonces: \(safereportid, drug_key, reaction_pt)

## 8. ¿Cuál es la diferencia entre receivedate, receiptdate y qde_period?

* receivedate: fecha historica de recepcion inicial del caso
* receiptdate: recha de recepcion de esta version especifica del reporte, corresponde casi siempre con el trimestre de extracto
* qde_period: el trimestre del archivo QDE del que priviene fisicamente el reporte  
En el 2025Q1 de 400,514 reportes en 400.511 casos el trismestre calculado a partir de \receiptdate coincidio con \qde_period  
El porcentaje de concordancias es 400,511/400,514 x 100 = 99.9993 %  
esto indic una corcondancia practicamente total, casi todos los reportes de una extracto trimesntral corresponde efectivamente a ese mismo trimestre segun \ receipdate, lo que valida usar \receiptdate como referencia temporal confiable.  Se decidio usar \analysis_date= receipdate(en vez de receivedate) porque corresponde a la fecha origianl del caso, representada en años antes, mientras que \receiptdate refleja cuando se recibio esta version del reporte, alineandose casi al periodo del extracto (qde_period). Esto hace mucho mas fiable como contruir un eje temporal coherente con los datos realmente disponibles en cada trimestre.|   


## 9. ¿Qué información proporciona safetyreportversion?

Indica cuantas actualizaciones ha reportado un caso desde su reporte original
* safetyreportversion = 1: es la primera version del reporte, sin actualizaciones posteriores
* safetyreportversion = 57 o 146: son casos que han sidp actualizados numerosa veces(seguimiento o correcciones/adiciones de informacion).  

Es importante investigar si un mismo safetyreportid reaparece en distintos trimestres porque, sino se controla al construir señales longitudinales un mismo caso podria contarse multiples veces en distintos periodos distorsionando el analisis de tendencias temporales. 

## 10. Describe el pipeline que se ha construido hasta este momento

Los archivos QDE se parte de los extractos trimestralkes descargados de la FDA (2025Q1 A 2026Q2) cada uno con su carpeta XML. La lectura incremental nos ayuda a extraer variables necesarias y liberar cada elemento de la memoria antes de continuar(**ET.iterparse()**). Para la extraccion de **safetyreportid** nos extrae e identifica sus variables principales(pasi, fecha, version, etc) para asi hacer el filtrado  de medicamentos usando **suspect**  de todos los \<**drug**> de reporte, se selecciona los que tienen **drugcharacterization = 1** para contruir la **drug_key** , se prioriza **activesubstancename** y se usa **medicinalproduct** como respaldo para unificar nombres comercionales bajo el mismo principio activo. Ahora hacemos la deduplicacion, se agrupa por **safetyreportid, drug_key** para que cada medicamento cuente una sola vez por reporte. Para terminar con pares medicamento-evento, donde incorporaremos las reacciones **reactionmeddrapt** para llegar a la unidad dinal **safetyreportid, drug_key, reaction_pt** que servira como base para calcular las señales de desproporcionalidad. 

**XML -> safetyreport -> drug/reaction -> drug_key -> limpieza -> deduplicacion -> (safetyreportid, drug-key, reaction_pt)**

** MAGALY BENAVIDES SANTIAGO**